# ED.03.03 DataFrames en Spark
Regina Zoé García Trejo - IDIA 222

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, avg, max, min, sum, count, desc, rank, trim, concat_ws, when, year, to_date
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("ED03-DataFrames").getOrCreate()

## Ejercicio 1: Creación y exploración

In [ ]:
datos = [("Ana", 28, "Ventas", 35000), ("Juan", 35, "IT", 45000), ("Maria", 42, "IT", 55000),
         ("Pedro", 31, "Ventas", 38000), ("Laura", 29, "Marketing", 40000)]
columnas = ["nombre", "edad", "departamento", "salario"]

# 1. Crear el DataFrame
df = spark.createDataFrame(datos, columnas)

# 2. Esquema y primeras 3 filas
df.printSchema()
df.show(3)

# 3. Total de empleados
print(f"Total de empleados: {df.count()}")

## Ejercicio 2: Filtrado

In [ ]:
# 1. Empleados de IT con salario > 40000
df.filter((col("departamento") == "IT") & (col("salario") > 40000)).show()

# 2. Empleados de Ventas o Marketing
df.filter((col("departamento") == "Ventas") | (col("departamento") == "Marketing")).show()

# 3. Empleados que NO son de IT
df.filter(~(col("departamento") == "IT")).show()

# 4. Empleados entre 30 y 40 anios
df.filter((col("edad") >= 30) & (col("edad") <= 40)).show()

## Ejercicio 3: Manipulación de columnas

In [ ]:
df3 = df.withColumn("salario_en_5_años", col("salario") + (col("salario") * 0.05 * 5))
df3 = df3.withColumnRenamed("nombre", "empleado")
df3.show()

## Ejercicio 4: Agregaciones

In [ ]:
# 1. Salario promedio por departamento
df.groupBy("departamento").agg(avg("salario").alias("salario_promedio")).show()

# 2. Salario máximo y mínimo por departamento
df.groupBy("departamento").agg(
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo")).show()

# 3. Empleados por departamento
df.groupBy("departamento").count().show()

# 4. Gasto total en salarios por departamento, de mayor a menor
df.groupBy("departamento").agg(sum("salario").alias("gasto_total")) \
  .orderBy(desc("gasto_total")).show()

## Ejercicio 5: Window Functions

In [ ]:
ventas = spark.createDataFrame(
    [("Enero", "ProdA", 100), ("Enero", "ProdB", 150), ("Febrero", "ProdA", 120)],
    ["mes", "producto", "ventas"])

ventana = Window.partitionBy("mes").orderBy(col("ventas").desc())
ventas.withColumn("ranking", rank().over(ventana)).show()

## Ejercicio 6: Joins

In [ ]:
personas = spark.createDataFrame([("Ana", 1), ("Luis", 2), ("Marta", 3)], ["Nombre", "DepID"])
departamentos = spark.createDataFrame([(1, "IT"), (2, "HR")], ["DepID", "NombreDep"])

personas.join(departamentos, "DepID", "left").select("Nombre", "DepID", "NombreDep").show()

Marta tiene DepID = 3, que no existe en el DataFrame de departamentos. Con un Left Join se conserva su fila, pero la columna NombreDep queda en `null`.

## Ejercicio 7: SQL puro

In [ ]:
df.createOrReplaceTempView("empleados")

spark.sql("""
    SELECT *
    FROM empleados
    WHERE departamento = 'IT' AND salario > 40000
""").show()

## Ejercicio 8: Funciones de string

In [ ]:
nombres = spark.createDataFrame([(" Juan ", "García"), ("Ana", " Pérez ")], ["nombre", "apellido"])

nombres_limpios = nombres.withColumn("nombre", trim(col("nombre"))) \
                         .withColumn("apellido", trim(col("apellido"))) \
                         .withColumn("Nombre_Completo", concat_ws(" ", col("nombre"), col("apellido")))
nombres_limpios.show()

## Proyecto: Análisis de ventas

In [ ]:
!mkdir -p DATOS
!wget "https://raw.githubusercontent.com/josepgarcia/datos/main/pdi_sales.csv" -P ./DATOS

**Paso 1.** Cargar el CSV con esquema inferido y cabeceras (separador `;`).

In [ ]:
df_ventas = spark.read.csv("DATOS/pdi_sales.csv", header=True, inferSchema=True, sep=";")

**Paso 2.** Esquema y número de filas.

In [ ]:
df_ventas.printSchema()
print(f"Total de filas: {df_ventas.count()}")

**Paso 3.** Ingresos totales por país, de mayor a menor.

In [ ]:
df_ventas.groupBy("Country").agg(sum("Revenue").alias("ingresos_totales")) \
         .orderBy(desc("ingresos_totales")).show()

**Paso 4.** Ventas de Alemania con más de 1 unidad vendida.

In [ ]:
alemania = df_ventas.filter((col("Country") == "Germany") & (col("Units") > 1))
alemania.show(10)
print(f"Filas: {alemania.count()}")

**Paso 5.** Código postal con mayor número de ventas (filas) en Alemania.

In [ ]:
df_ventas.filter(col("Country") == "Germany") \
         .groupBy("Zip").count() \
         .orderBy(desc("count")).show(1)

**Paso 6.** Columna que clasifica la venta: `Alta` si Revenue > 500, si no `Normal`.

In [ ]:
df_clasif = df_ventas.withColumn("Clasificacion", when(col("Revenue") > 500, "Alta").otherwise("Normal"))
df_clasif.select("ProductID", "Revenue", "Clasificacion").show(10)

**Paso 7 (Bonus).** Media de ingresos por año (la fecha viene como texto M/D/AAAA).

In [ ]:
df_fecha = df_ventas.withColumn("Fecha", to_date(col("Date"), "M/d/yyyy")) \
                    .withColumn("Anio", year(col("Fecha")))

df_fecha.groupBy("Anio").agg(avg("Revenue").alias("ingreso_medio")).orderBy("Anio").show()